# HE²AT synthetic southern Africa: a first run in Python

Every value in this dataset is simulated. Results are about methods, not about real people or places. Read `DATA_USE.md` in the repo before you start.

Run each cell in turn with **Shift + Enter**.

## 1. Get the repo

The repo is public, so Colab can clone it directly. No GitHub account or token is needed.

In [ ]:
import os, sys
REPO = 'he2at-synthetic-southern-africa'
if 'google.colab' in sys.modules and not os.path.exists(REPO):
    !git clone -q https://github.com/Wits-Planetary-Health-Research/{REPO}.git
if os.path.exists(REPO):
    %cd {REPO}
!ls

Colab already has pandas, pyarrow, statsmodels and matplotlib. This just makes sure.

In [ ]:
!pip install -q -r requirements.txt

## 2. Load the pregnancies

In [ ]:
sys.path.insert(0, 'examples')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from common import participants, district_week_exposure, add_anomaly, person_weeks, attach_lags, DATA, DAY0

df = participants()
df.shape

In [ ]:
df.head()   # analysis-ready names; the raw file uses the HEAT Master Codebook names

## 3. A first look at the outcomes

In [ ]:
print(f'{len(df):,} pregnancies in {df.district_id.nunique()} districts')
print(f'Births from {df.birth_date.min().date()} to {df.birth_date.max().date()}')
df[['preterm', 'very_preterm', 'stillbirth', 'low_birth_weight']].mean().round(3)

In [ ]:
df.groupby('country').preterm.agg(['mean', 'size']).round(3)

In [ ]:
df.groupby('outdoor_work_pregnancy').preterm.agg(['mean', 'size']).round(3)

## 4. Map the preterm share by district

In [ ]:
import json
from matplotlib.collections import PatchCollection
from matplotlib.patches import Polygon

rate = df.groupby('district_id').preterm.mean()
gj = json.load(open(DATA / 'districts.geojson'))
patches = [Polygon(f['geometry']['coordinates'][0], closed=True) for f in gj['features']]
vals = [100 * rate[f['properties']['district_id']] for f in gj['features']]

fig, ax = plt.subplots(figsize=(6, 6))
pc = PatchCollection(patches, cmap='Greys', edgecolor='white', linewidth=0.3)
pc.set_array(vals); ax.add_collection(pc)
ax.autoscale(); ax.set_aspect('equal')
plt.colorbar(pc, ax=ax, label='Preterm births (%)', fraction=0.04)
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
plt.show()

## 5. The weather: weekly district temperature from the gridded product

In [ ]:
ex = district_week_exposure()
ex.head()

In [ ]:
one = ex[ex.district_id == 'ZAF_D01']
plt.figure(figsize=(9, 3))
plt.plot(one.week_start, one.tmax, color='0.3', lw=0.8)
plt.ylabel('Weekly mean tmax (°C)'); plt.title('ZAF_D01')
plt.show()

## 6. Build your own surface from the stations

The simplest surface: inverse distance weighting of station tmax, corrected for height, averaged to districts. Swap in SPDE, kriging or a neural network here.

In [ ]:
cells = pd.read_csv(DATA / 'district_cells_0p25deg.csv')
st = pd.read_csv(DATA / 'stations.csv')
obs = pd.read_parquet(DATA / 'weather_stations_daily.parquet')
Y = obs.pivot(index='date', columns='station_id', values='tmax')[st.station_id]
Y.shape   # days x stations

In [ ]:
LAPSE = 0.0055   # °C per metre

def km(lon, lat):
    return np.c_[(np.asarray(lon) - 25) * 111.32 * np.cos(np.deg2rad(-20)), (np.asarray(lat) + 20) * 110.57]

D = np.linalg.norm(km(cells.lon, cells.lat)[:, None, :] - km(st.lon, st.lat)[None, :, :], axis=2)
W0 = 1.0 / np.maximum(D, 5.0) ** 2
Ysl = Y.values + LAPSE * st.elev_m.values[None, :]
ok = ~np.isnan(Ysl)
field = (np.nan_to_num(Ysl) @ W0.T) / (ok.astype(float) @ W0.T) - LAPSE * cells.elev_m.values[None, :]
field.shape   # days x cells

In [ ]:
daily = pd.DataFrame(field, index=Y.index).T.groupby(cells.district_id.values).mean().T
weekly_idw = daily.groupby((pd.to_datetime(daily.index) - DAY0).days // 7).mean()
weekly_grid = ex.pivot(index='wk', columns='district_id', values='tmax')
diff = (weekly_idw - weekly_grid).stack()
print(f'IDW minus gridded, district-week tmax: mean {diff.mean():.2f} °C, SD {diff.std():.2f} °C')

## 7. Is heat related to preterm birth?

A discrete-time hazard model: one row per pregnancy per gestational week from 22 to 36, with temperature in the weeks before. This builds about 2.8 million rows and takes a minute or two.

In [ ]:
ex = add_anomaly(ex, 'tmax')
ex['tmax_change'] = ex.tmax_anom - ex.groupby('district_id').tmax_anom.shift(1)

cols = ['id', 'district_id', 'lmp_date', 'birth_date', 'ga_days', 'age_years', 'ses_grade', 'hiv_positive',
        'anaemia', 'hypertensive_disorder', 'outdoor_work_pregnancy', 'infant_sex']
pw = person_weeks(df[cols])
pw = attach_lags(pw, ex, 'tmax_anom')
pw = attach_lags(pw, ex, 'tmax_change')
pw = pw.dropna(subset=[c for c in pw if c.startswith('tmax_')])
print(f'{len(pw):,} person-weeks, {pw.event.sum():,} preterm births')

In [ ]:
w = np.array([0.40, 0.28, 0.17, 0.10, 0.05])   # lag weights, try your own
pw['heat'] = sum(w[l] * pw[f'tmax_anom_lag{l}'] for l in range(5))
pw['change'] = sum(w[l] * pw[f'tmax_change_lag{l}'] for l in range(5))

X = pd.get_dummies(pw.gest_week, prefix='gw', drop_first=True, dtype=float)
X['age_35plus'] = (pw.age_years > 35).astype(float)
X['age_under22'] = (pw.age_years < 22).astype(float)
X = X.join(pd.get_dummies(pw.ses_grade, prefix='ses', drop_first=True, dtype=float))
for c in ['hiv_positive', 'anaemia', 'hypertensive_disorder', 'outdoor_work_pregnancy']:
    X[c] = pw[c].astype(float)
X['male'] = (pw.infant_sex == 'male').astype(float)

In [ ]:
import statsmodels.api as sm

def fit(cols):
    m = sm.Logit(pw.event, sm.add_constant(X.join(pw[cols]))).fit(disp=0)
    ci = np.exp(m.conf_int().loc[cols])
    return pd.DataFrame({'OR': np.exp(m.params[cols]), 'lo': ci[0], 'hi': ci[1]}).round(3)

fit(['heat'])   # odds ratio per °C of lag-weighted tmax anomaly

In [ ]:
fit(['heat', 'change'])   # add week-on-week change

## Your turn

Replace the gridded exposure in step 7 with your own surface from step 6, and see whether the answer changes. Example results to beat are in `docs/method_comparison.md`.